In [1]:
import pandas as pd
import numpy as np
import datetime

import sys
sys.path.append("/Users/derekdewald/Documents/Python/Github_Repo/d_py_functions")

# Import Google Sheet Notes
notes_df = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vSQF2lNc4WPeTRQ_VzWPkqSZp4RODFkbap8AqmolWp5bKoMaslP2oRVVG21x2POu_JcbF1tGRcBgodu/pub?output=csv')
# Import Google Sheet Definition
definition_df = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vQq1-3cTas8DCWBa2NKYhVFXpl8kLaFDohg0zMfNTAU_Fiw6aIFLWfA5zRem4eSaGPa7UiQvkz05loW/pub?output=csv')
# Import Manual File, which is generated from Function, which uses OBJECTS_MANUAL.PY  - Possibly to be replaced by Github file..
manual_object_df= pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/object_manual_published.xlsx')


In [14]:
from objects_manual import object_dict

object_dict.keys()
print(object_dict['template_doc_string']['python_object'])



    Definition of Function

    Parameters:
        List of Parameters

    Returns:
        Object Type

    date_created:02-Jul-26
    date_last_modified: 02-Jul-26
    classification:TBD
    sub_classification:TBD
    usage:
        Example Function Call



In [19]:
def generate_knowledgebase(
    notes_df=None,
    definition_df=None,
    manual_object_df=None,
    export_location=None):

    '''
    Process Utilized to Combine Notes/ Definitions and Logic into Knowledge Base, which is utilized to Create, Processes, Parameters.

    Parameters:
        notes_df (dataframe): Dataframe containing Notes from Google. Default is none and it will pull directly from Google.
        definition_df (dataframe): Dataframe containing Definitions from Google. Default is none and it will pull directly from Google.
        manual_object_df (dataframe): Dataframe containining Dataframe of Parameters, generated from Python Process _____, which converts lists in objects_manual.py.
        export_location(str): Name of File to export excel file to. If Blank, returns nothing

    Returns:
        dataframe
        excel_file

    date_created:02-Jul-26
    date_last_modified: 02-Jul-26
    classification:TBD
    sub_classification:TBD
    usage:
        d = generate_knowledgebase(notes_df,definition_df,manual_object_df)
        d = generate_knowledgebase()
 
    '''


    
    if len(notes_df)==0:
        notes_df = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vSQF2lNc4WPeTRQ_VzWPkqSZp4RODFkbap8AqmolWp5bKoMaslP2oRVVG21x2POu_JcbF1tGRcBgodu/pub?output=csv')

    if len(definition_df)==0:
        definition_df = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vQq1-3cTas8DCWBa2NKYhVFXpl8kLaFDohg0zMfNTAU_Fiw6aIFLWfA5zRem4eSaGPa7UiQvkz05loW/pub?output=csv')

    if len(manual_object_df)==0:
        manual_object_df = pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/object_manual_published.xlsx')

    # Combine Notes from Google and Notes Extract from Manual List
    consolidated_notes = pd.concat([notes_df,manual_object_df.drop('Order',axis=1)])
    
    # Take all Processes from Notes, so we can extract Definitions
    process_list = consolidated_notes['Process'].unique().tolist()
    
    # Extract Definitions to include into Consolidated Notes.
    notes_from_def = definition_df[definition_df['Process'].isin(process_list)][['Process','Categorization','Word','Definition']]
    
    # Create a Consolidated notes list (Before Incorporating Definitions).
    
    final_df = pd.concat([
        consolidated_notes,
        notes_from_def])
    
    # Merge in Sub Processes
    word_list = final_df['Word'].unique().tolist()
    sub_processes = final_df[(final_df['Process'].isin(word_list))]
    #sub_processes = sub_processes.drop(['Categorization'],axis=1).rename(columns={'Word':"Categorization",'Process':'Word','Definition':"Definition"})
    # Include Process.
    
    sub_processes['Definition'] = sub_processes['Word'].fillna("") + ": " + sub_processes['Definition'].fillna('')
    sub_processes['Word'] = sub_processes['Process']
    sub_processes.drop('Process',axis=1,inplace=True)
    sub_processes = sub_processes.merge(final_df[['Word','Process']],on='Word',how='left')
    
    
    final_df = pd.concat([
        final_df,
        sub_processes])
    
    final_df = final_df.merge(definition_df[['Word','Definition']].rename(columns={'Definition':'Definition1'}),on='Word',how='left')
    final_df['Definition'] = np.where(final_df['Definition'].notnull(),final_df['Definition'],final_df['Definition1'])
    final_df.drop('Definition1',axis=1,inplace=True)
    #final_df.reset_index(inplace=True)
    #final_df.rename(columns={'index':'default_order'},inplace=True)
    
    # Sorting
    # Process - Has to be Alphabetical. Do not actually Care Order.
    # Categorization - Based on Manually defined Order, imported from Objects_Manual.
    # Words Matter when they are part of a process, generally when not then alphabetically preferred. 
    
    # Merge in Process  Filter
    proc_filter = manual_object_df[(manual_object_df['Process']=='Notes')&(manual_object_df['Categorization']=='Filter Order')][['Word','Order']].reset_index(drop=True).rename(columns={'Word':"Categorization",'Order':"proc_order"})
    final_df = final_df.merge(proc_filter,on='Categorization',how='left')
    
    # Merge in Word Order.
    final_df = final_df.merge(manual_object_df.drop('Categorization',axis=1).rename(columns={'Order':'word_order'}),on=['Process','Word'],how='left')
    
    final_df.sort_values(['Process','word_order','proc_order','Word'],inplace=True)
    final_df.drop(['word_order','proc_order'],axis=1,inplace=True)

    if export_location:
        final_df.to_excel(export_location,index=False)
    
    
    return 

    
d = generate_knowledgebase(notes_df,definition_df,manual_object_df,'/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/knowledge_base.xlsx')
    

In [18]:
import os
#os.getcwd()
os.listdir('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/')

['consolidated_notes.csv', 'object_manual_published.xlsx']

In [7]:
d.groupby(['Process','Categorization']).size()

Process                     Categorization
Behavioural Economics       Concept           14
                            Information        6
Data Collection             Guidance           3
Data Dictionary             Column List       10
Data Preparation            Definition         8
                            Process Step       3
                            Regularization     5
Documentation               Reference List     7
Feature Engineering         Guidance           3
Goal Setting                Requirement        5
Machine Learning Lifecycle  Definition         8
                            Guidance          10
                            Process Step      19
                            Regularization     5
                            Requirement       10
Notes                       Filter Order       9
Problem Definition          Guidance           4
                            Requirement        5
Problem Solving Framework   Guidance           7
                          

In [9]:
d[d['Process']=='Machine Learning Lifecycle']

,Process,Categorization,Word,Definition
64,Machine Learning Lifecycle,Process Step,Goal Setting,"Thought process utilized to establish a clear,..."
121,Machine Learning Lifecycle,Requirement,Goal Setting,Specific: Describes what will be accomplished ...
123,Machine Learning Lifecycle,Requirement,Goal Setting,Measurable: Includes a clear way to track prog...
125,Machine Learning Lifecycle,Requirement,Goal Setting,"Achievable: Realistic given available time, re..."
127,Machine Learning Lifecycle,Requirement,Goal Setting,"Relevant: Aligns with broader priorities, such..."
129,Machine Learning Lifecycle,Requirement,Goal Setting,Time Bound: Includes a clear deadline or timef...
65,Machine Learning Lifecycle,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th..."
141,Machine Learning Lifecycle,Guidance,Problem Definition,Understand KPIs which are meaningful.: Attempt...
143,Machine Learning Lifecycle,Guidance,Problem Definition,What Lens is Problem Defined?: Must be careful...
145,Machine Learning Lifecycle,Guidance,Problem Definition,Stakeholder Investment?: How Invested are stak...


In [3]:
d[d['Process']=='Problem Solving Framework']

,Process,Categorization,Word,Definition
45,Problem Solving Framework,Process Step,Goal Setting,"Thought process utilized to establish a clear,..."
120,Problem Solving Framework,Requirement,Goal Setting,Specific: Describes what will be accomplished ...
122,Problem Solving Framework,Requirement,Goal Setting,Measurable: Includes a clear way to track prog...
124,Problem Solving Framework,Requirement,Goal Setting,"Achievable: Realistic given available time, re..."
126,Problem Solving Framework,Requirement,Goal Setting,"Relevant: Aligns with broader priorities, such..."
128,Problem Solving Framework,Requirement,Goal Setting,Time Bound: Includes a clear deadline or timef...
46,Problem Solving Framework,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th..."
140,Problem Solving Framework,Guidance,Problem Definition,Understand KPIs which are meaningful.: Attempt...
142,Problem Solving Framework,Guidance,Problem Definition,What Lens is Problem Defined?: Must be careful...
144,Problem Solving Framework,Guidance,Problem Definition,Stakeholder Investment?: How Invested are stak...


In [24]:
word_list = final_df['Word'].unique().tolist()
sub_processes = final_df[(final_df['Process'].isin(word_list))]


In [25]:
# Combine Notes from Google and Notes Extract from Manual List
consolidated_notes = pd.concat([notes_df,manual_object_df.drop('Order',axis=1)])

# Take all Processes from Notes, so we can extract Definitions
process_list = consolidated_notes['Process'].unique().tolist()

# Extract Definitions to include into Consolidated Notes.
notes_from_def = definition_df[definition_df['Process'].isin(process_list)][['Process','Categorization','Word','Definition']]

# Create a Consolidated notes list (Before Incorporating Definitions).

final_df = pd.concat([
    consolidated_notes,
    notes_from_def])

# Merge in Sub Processes
word_list = final_df['Word'].unique().tolist()
sub_processes = final_df[(final_df['Process'].isin(word_list))]
#sub_processes = sub_processes.drop(['Categorization'],axis=1).rename(columns={'Word':"Categorization",'Process':'Word','Definition':"Definition"})
# Include Process.

sub_processes['Definition'] = sub_processes['Word'].fillna("") + ": " + sub_processes['Definition'].fillna('')
sub_processes['Word'] = sub_processes['Process']
sub_processes.drop('Process',axis=1,inplace=True)
sub_processes = sub_processes.merge(final_df[['Word','Process']],on='Word',how='left')


final_df = pd.concat([
    final_df,
    sub_processes])

final_df = final_df.merge(definition_df[['Word','Definition']].rename(columns={'Definition':'Definition1'}),on='Word',how='left')
final_df['Definition'] = np.where(final_df['Definition'].notnull(),final_df['Definition'],final_df['Definition1'])
final_df.drop('Definition1',axis=1,inplace=True)
#final_df.reset_index(inplace=True)
#final_df.rename(columns={'index':'default_order'},inplace=True)

# Sorting
# Process - Has to be Alphabetical. Do not actually Care Order.
# Categorization - Based on Manually defined Order, imported from Objects_Manual.
# Words Matter when they are part of a process, generally when not then alphabetically preferred. 

# Merge in Process  Filter
proc_filter = manual_object_df[(manual_object_df['Process']=='Notes')&(manual_object_df['Categorization']=='Filter Order')][['Word','Order']].reset_index(drop=True).rename(columns={'Word':"Categorization",'Order':"proc_order"})
final_df = final_df.merge(proc_filter,on='Categorization',how='left')

# Merge in Word Order.
final_df = final_df.merge(manual_object_df.drop('Categorization',axis=1).rename(columns={'Order':'word_order'}),on=['Process','Word'],how='left')

final_df.sort_values(['Process','word_order','proc_order','Word'],inplace=True)

In [26]:
final_df['Process'].value_counts()

Process
Machine Learning Lifecycle    24
Problem Solving Framework     24
Behavioural Economics         20
Data Dictionary               10
Notes                          9
Documentation                  7
Goal Setting                   5
Feature Engineering            3
Organization - Definition      2
Name: count, dtype: int64

In [27]:
final_df[final_df['Word']=='Feature Engineering']

,Process,Categorization,Word,Definition,proc_order,word_order
56,Machine Learning Lifecycle,Process Step,Feature Engineering,Process of creating new input variables that b...,3.0,6.0
91,Machine Learning Lifecycle,Guidance,Feature Engineering,Ratios or Observed Values: What ratios or prop...,5.0,6.0
92,Machine Learning Lifecycle,Guidance,Feature Engineering,Include Interactions: What combinations or int...,5.0,6.0
93,Machine Learning Lifecycle,Guidance,Feature Engineering,How does Time Impact?: What time-based changes...,5.0,6.0


In [28]:
final_df[final_df['Process']=='Machine Learning Lifecycle']

,Process,Categorization,Word,Definition,proc_order,word_order
51,Machine Learning Lifecycle,Process Step,Goal Setting,Process/ Thought process utilized to establish...,3.0,1.0
95,Machine Learning Lifecycle,Requirement,Goal Setting,Specific: Describes what will be accomplished ...,NaN,1.0
97,Machine Learning Lifecycle,Requirement,Goal Setting,Measurable: Includes a clear way to track prog...,NaN,1.0
99,Machine Learning Lifecycle,Requirement,Goal Setting,"Achievable: Realistic given available time, re...",NaN,1.0
101,Machine Learning Lifecycle,Requirement,Goal Setting,"Relevant: Aligns with broader priorities, such...",NaN,1.0
103,Machine Learning Lifecycle,Requirement,Goal Setting,Time Bound: Includes a clear deadline or timef...,NaN,1.0
52,Machine Learning Lifecycle,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th...",3.0,2.0
53,Machine Learning Lifecycle,Process Step,Data Collection,"Data collection involves identifying, sourcing...",3.0,3.0
54,Machine Learning Lifecycle,Process Step,Data Preparation,Data preparation focuses on cleaning and struc...,3.0,4.0
55,Machine Learning Lifecycle,Process Step,Exploratory Data Analysis,Foundational step in ensuring the success of a...,3.0,5.0


In [37]:
d[d['Process']=='Problem Solving Framework']

,Process,Categorization,Word,Definition
32,Problem Solving Framework,Process Step,Goal Setting,Process/ Thought process utilized to establish...
94,Problem Solving Framework,Requirement,Goal Setting,Specific: Describes what will be accomplished ...
96,Problem Solving Framework,Requirement,Goal Setting,Measurable: Includes a clear way to track prog...
98,Problem Solving Framework,Requirement,Goal Setting,"Achievable: Realistic given available time, re..."
100,Problem Solving Framework,Requirement,Goal Setting,"Relevant: Aligns with broader priorities, such..."
102,Problem Solving Framework,Requirement,Goal Setting,Time Bound: Includes a clear deadline or timef...
33,Problem Solving Framework,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th..."
34,Problem Solving Framework,Process Step,Current State Assessment,NaN
35,Problem Solving Framework,Process Step,Data Collection,"Data collection involves identifying, sourcing..."
36,Problem Solving Framework,Process Step,Root Cause Analysis,NaN


In [113]:
final_df.groupby(['Process','Categorization']).size()

Process                     Categorization
Behavioural Economics       Definition        14
                            Information        6
Data Dictionary             Column List       10
Documentation               Reference List     7
Feature Engineering         General Note       3
Machine Learning Lifecycle  Guidance           3
                            Process Step      16
Notes                       Filter Order       9
Organization - Definition   Guidance           2
Problem Solving Framework   Process Step      19
dtype: int64

In [109]:
final_df.to_excel('test.xlsx')

In [ ]:
# In addition to including Adding Words Associate with the Definition, add the definition itself.
process_df = pd.DataFrame(process_list,columns=['Process'])
process_df['Word'] = process_df['Process']
process_df['Categorization'] = "Process Definition"

final_df = pd.concat([

    final_df,
    process_df
    
])

final_df = final_df.merge(definition_df[['Word','Definition']].rename(columns={'Definition':'Definition1'}),on='Word',how='left')
final_df['Definition'] = np.where(final_df['Definition'].notnull(),final_df['Definition'],final_df['Definition1'])
final_df.drop('Definition1',axis=1,inplace=True)
final_df.reset_index(inplace=True)
final_df.rename(columns={'index':'default_order'},inplace=True)

final_df = final_df.merge(cat_filter,on='Categorization',how='left')
final_df.sort_values(['Process','CAT_ORDER','default_order'],inplace=True)

,Process,Categorization,Word,Definition,Categorization_,Definition_
47,Machine Learning Lifecycle,Process,Goal Setting,NaN,NaN,NaN
48,Machine Learning Lifecycle,Process,Problem Definition,NaN,NaN,NaN
49,Machine Learning Lifecycle,Process,Data Collection,NaN,NaN,NaN
50,Machine Learning Lifecycle,Process,Data Preparation,NaN,NaN,NaN
51,Machine Learning Lifecycle,Process,Exploratory Data Analysis,NaN,NaN,NaN
52,Machine Learning Lifecycle,Process,Feature Engineering,NaN,Requirement,What ratios or proportions could better repres...
53,Machine Learning Lifecycle,Process,Feature Engineering,NaN,Requirement,What combinations or interactions might reflec...
54,Machine Learning Lifecycle,Process,Feature Engineering,NaN,Requirement,What time-based changes or aggregates could ca...
55,Machine Learning Lifecycle,Process,Feature Selection,NaN,NaN,NaN
56,Machine Learning Lifecycle,Process,Model Selection,NaN,NaN,NaN


In [53]:
# Exclude Process Definition or literally every word will be included, only want not Lead Processes, but sub-components.

sub_processes

,Word,Categorization_,Definition_
4,Feature Engineering,Requirement,What ratios or proportions could better repres...
5,Feature Engineering,Requirement,What combinations or interactions might reflec...
6,Feature Engineering,Requirement,What time-based changes or aggregates could ca...


In [56]:
final_df[final_df['Process']=='Feature Engineering']

,Process,Categorization,Word,Definition,Categorization_,Definition_
4,Feature Engineering,Process Step,Requirement,What ratios or proportions could better repres...,NaN,NaN
5,Feature Engineering,Process Step,Requirement,What combinations or interactions might reflec...,NaN,NaN
6,Feature Engineering,Process Step,Requirement,What time-based changes or aggregates could ca...,NaN,NaN


In [30]:
final_df.to_excel('delete.xlsx')

In [93]:
def consolidated_knowledge_base(export_location=None,
                                ordered_cat= ['Problem Framework Order','Definition Categorization Order']):

    '''
    Process to Integration Notes, Definitions and Manual Objects Lists into D Knowledge Base, D Processes and D Default Lists.
    
    Idea is to prevent redudant data input by controlling structure and iteratively being able to easily build upon and maintaine the structure. 
    

    Step 1. Import Data and Required Control Definitions.
    Step 2. Import Definitions
    
    Step 2.
    Step 3. Merge in definition_df (Except General Definitions, which are only used to populate Information)
    Step 4.
    

    ############### ONLY WANT TO INCLUDE DEFINITIONS WHICH MEET PROCESS AND CATEGORIZATION DEFINITIONS.
    ############### How do I want to Maintain Specific Processes.
    
    I want to merge in definitions, but not all.


    #### KEEP NOTES DISTINCT FROM Knowledge Base, Process and Check List.

    
    Parameters:
        export_Location (str): If populated, location where .csv file will be saved.

    Returns:
        Object Type

    date_created:12-Jan-26
    date_last_modified: 12-Jun-26
    classification:TBD
    sub_classification:TBD
    usage:
        notes_df = generate_dictionary()
        
    '''
    
    if not export_location:
        export_location = '/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/consolidated_notes.csv'
    
    list_ = ['Process','Categorization','Word','Definition']

    # Download Data
    definition_df = pd.read_csv(links['google_definition_csv'])
    notes_df = pd.read_csv(links['google_notes_csv'])

    # Limit to Required Information
    definition_df = definition_df[list_].copy()
    notes_df = notes_df[list_].copy()

    # Populate Missing Definitions in Notes
    notes_df = notes_df.merge(definition_df[['Word','Definition']].rename(columns={'Definition':'Definition1'}),on='Word',how='left')
    notes_df['Definition'] = np.where(notes_df['Definition'].notnull(),notes_df['Definition'],notes_df['Definition1'])
    notes_df.drop('Definition1',axis=1,inplace=True)

    # Define Order 
    order = notes_df[notes_df['Categorization'].isin(ordered_cat)]['Word'].tolist()
    order_df = pd.DataFrame(order,columns=['Categorization']).reset_index().rename(columns={'index':'ORDER'})

    # Remove General Definitions
    definition_df1 = definition_df[definition_df['Process']!='General Definition'].copy()

    
    # Consolidate Dataframe
    final_df = pd.concat([
        notes_df,
        definition_df1]).merge(order_df.rename(columns={"ORDER":'ORDER_COLUMN'}),on='Categorization',how='left').merge(order_df.rename(columns={'Categorization':'Word',"ORDER":'ORDER_WORD'}),on='Word',how='left').reset_index(drop=True)

    return final_df.sort_values(['Process','ORDER_COLUMN','ORDER_WORD','Word']).reset_index(drop=True),order_df

final_df,order_df = consolidated_knowledge_base()
final_df

,Process,Categorization,Word,Definition,ORDER_COLUMN,ORDER_WORD
0,Behavioural Economics,Definition,Anchoring Effect,A cognitive bias where people rely too heavily...,0.0,NaN
1,Behavioural Economics,Definition,Base Rate Fallacy,People ignore or undervalue general statistica...,0.0,NaN
2,Behavioural Economics,Definition,Biased Assimilation,Tendency to interpret information in a way tha...,0.0,NaN
3,Behavioural Economics,Definition,Blind spot bias,Cognitive bias where people are unaware of the...,0.0,NaN
4,Behavioural Economics,Definition,Bureaucratic Politics,Organization decision making paradigm in which...,0.0,NaN
5,Behavioural Economics,Definition,Confirmation Bias,Also known as Selective Perception. What peopl...,0.0,NaN
6,Behavioural Economics,Definition,Fact Believability,Facts about this epidemic that have lasted a f...,0.0,NaN
7,Behavioural Economics,Definition,Facts Vs Statistics,More likely to die in accident in evening than...,0.0,NaN
8,Behavioural Economics,Definition,Fast statistics,"Immoderate, intuitive, visceral, and powerful.",0.0,NaN
9,Behavioural Economics,Definition,Illusion of Asymmetric Insights,The Belief that we know others better than the...,0.0,NaN


In [90]:
final_df.to_excel('TEST.xlsx',index=False)

In [88]:
process_list = notes_df['Process'].unique().tolist()
process_list.extend([x for x in definition_df['Process'].unique() if (x not in process_list)&(pd.notna(x))])

process_list

['Behavioural Economics',
 'Organization',
 'Problem Framework',
 'General Definition']

In [86]:
process_list = notes_df['Process'].unique().tolist()
process_list.extend([x for x in definition_df['Process'].unique() if (x not in process_list)&(pd.notna(x))])
process_map = {x:count+0 for count,x in enumerate(process_list)}

In [ ]:
# # Step 1. Create a list of unique Processes from Notes.
# process_list = notes_df['Process'].unique().tolist()
# process_list.extend([x for x in definition_df['Process'].unique() if (x not in process_list)&(pd.notna(x))])
# process_map = {x:count+0 for count,x in enumerate(process_list)}

# # Step 2 Create a Unique Classification List
# categorization_list = [
#     'Definition','Guiding Principle','Consideration','Process Step','Procedure','Expected Outcomes','Parameter','Algorithm']

# categorization_list.extend([x for x in notes_df['Categorization'].unique() if (x not in categorization_list)&(pd.notna(x))])
# categorization_list.extend([x for x in definition_df['Categorization'].unique() if (x not in categorization_list)&(pd.notna(x))])
# categorization_map = {x:count+0 for count,x in enumerate(categorization_list)}

# final_df = pd.concat([notes_df,definition_df])
# final_df['PRO_ORDER'] = final_df['Process'].map(process_map)
# final_df['CAT_ORDER'] = final_df['Categorization'].map(categorization_map)

# final_df = final_df.sort_values(['PRO_ORDER','CAT_ORDER'])
# final_df.drop(['PRO_ORDER','CAT_ORDER'],axis=1,inplace=True)

# if export_location:
#     final_df.to_csv(export_location,index=False)


In [72]:
definition_df = pd.read_csv(links['google_definition_csv'])
notes_df = pd.read_csv(links['google_notes_csv'])

list_ = ['Process','Categorization','Word','Definition']

definition_df = definition_df[list_].copy()
notes_df = notes_df[list_].copy()


In [73]:
print(len(definition_df))
print(len(notes_df))

32
27


In [74]:
notes_df.merge(definition_df[['Word','Definition']].rename(columns={'Definition':'Definition1'}),on='Word',how='left')

,Process,Categorization,Word,Definition,Definition1
0,Behavioural Economics,Consideration,What Meaning Does a KPI actually have?,Before we figure out whether nurses have had a...,NaN
1,Behavioural Economics,Consideration,Organizations,Great at solving Problems. Bad at diagnosing ...,NaN
2,Behavioural Economics,Consideration,Think about the Medium,"Weekly, daily, hourly—the metronome of the new...",NaN
3,Behavioural Economics,Consideration,Unintended Consequences KPIs,Must consider what adverse effects one might c...,NaN
4,Organization,Definition Categorization Order,Definition,NaN,NaN
5,Organization,Definition Categorization Order,Guiding Principle,NaN,NaN
6,Organization,Definition Categorization Order,Consideration,NaN,NaN
7,Organization,Definition Categorization Order,Process Step,NaN,NaN
8,Organization,Definition Categorization Order,Procedure,NaN,NaN
9,Organization,Definition Categorization Order,Expected Outcomes,NaN,NaN


In [28]:
# Define priority order 

notes_df[notes_df['Categorization']=='Definition Categorization Order']

,Process,Categorization,Word,Definition
13,ML Model,Guiding Principle,Machine Learning,Using a process to enable computers to iterati...
14,ML Model,Guiding Principle,Optimization,Optimization is the process of finding the bes...
15,ML Model,Pipeline Stage,Model Summarization / Model Interpretability,Model summarization and interpretability aim t...
16,ML Model,Pipeline Stage,Monitoring,Monitoring tracks model performance and behavi...
26,ML Model,Algorithm,Ada,Boosting technique that combines multiple weak...
...,...,...,...,...
371,ML Model,Algorithm,Hard Voting,Hard voting is an ensemble decision strategy w...
372,ML Model,Algorithm,Soft Voting,Soft voting is an ensemble technique where cla...
403,ML Model,Model Type,Assessment,Assessment refers to techniques that do not le...
432,ML Model,Algorithm,Naive Bayes,Naive Bayes is a family of probabilistic machi...


In [26]:
definition_df['Process'].unique()

array(['Data Preparation', 'Model Evaluation', 'ML Model', 'Mathematics',
       'Algorithm Classification', 'Organization', 'Tooling', 'Training',
       'Validation', 'ML Definitions', 'Behavioural Economics', 'Theorem',
       'Data Engineering', 'Graph Theory', 'Machine Learning Operations',
       'Model Type', 'Generative AI', 'Langchain', 'Python', 'Clustering',
       nan, 'General Definition'], dtype=object)

In [27]:
definition_df['Categorization'].unique()

array(['Definition', 'Regularization', 'Feature Selection',
       'Guiding Principle', 'Pipeline Stage', 'Concept', 'Property',
       'Theorem', 'Algorithm', 'Function', 'Model Type', 'Procedure',
       'Model Architecture', 'Optimizer', 'Evaluation', 'Semantic Type',
       'Functional Role', 'Spark', 'Hadoop', 'Paradigm',
       'Automating Python', 'Parameter', 'Tooling', 'Model Evaluation',
       'Processing', 'Formula', nan], dtype=object)

In [22]:
definition_df.head()

,Process,Categorization,Word,Definition
0,Data Preparation,Definition,Baseline,Baseline represents the typical or expected be...
1,Data Preparation,Definition,Dimension,A Dimension is a conceptual axis of behavior o...
2,Data Preparation,Definition,Implementation,An Implementation is a specific measurable rep...
3,Data Preparation,Definition,Momentum,Momentum represents the direction and rate of ...
4,Data Preparation,Definition,Parameter,A Parameter is a tuning choice that controls h...


In [23]:
notes_df.head()

,Process,Categorization,Word,Definition
0,ML Project,Process Step,Goal Setting,NaN
1,ML Project,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th..."
2,ML Project,Process Step,Data Collection,"Data collection involves identifying, sourcing..."
3,ML Project,Process Step,Data Preparation,Data preparation focuses on cleaning and struc...
4,ML Project,Process Step,Exploratory Data Analysis,Foundational step in ensuring the success of a...


In [20]:
notes_df.merge(definition

{'google_definition_csv': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQq1-3cTas8DCWBa2NKYhVFXpl8kLaFDohg0zMfNTAU_Fiw6aIFLWfA5zRem4eSaGPa7UiQvkz05loW/pub?output=csv',
 'google_notes_csv': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSQF2lNc4WPeTRQ_VzWPkqSZp4RODFkbap8AqmolWp5bKoMaslP2oRVVG21x2POu_JcbF1tGRcBgodu/pub?output=csv',
 'google_word_quote': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTjXiFjpGgyqWDg9RImj1HR_BeriXs4c5-NSJVwQFn2eRKksitY46oJT0GvVX366LO-m1GM8znXDcBp/pub?gid=1117793378&single=true&output=csv',
 'google_daily_activities': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTjXiFjpGgyqWDg9RImj1HR_BeriXs4c5-NSJVwQFn2eRKksitY46oJT0GvVX366LO-m1GM8znXDcBp/pub?gid=472900611&single=true&output=csv',
 'technical_notes': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSnwd-zccEOQbpNWdItUG0qXND5rPVFbowZINjugi15TdWgqiy3A8eMRhbmSMBiRhHt1Qsry3E8tKY8/pub?output=csv',
 'd_links': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSwDznLz-GKgWFT1uN0XZYm3bsos899I9MS-pSv

In [25]:
# Step 1. Create a list of unique Processes from Notes.
process_list = notes_df['Process'].unique().tolist()
process_list.extend([x for x in definition_df['Process'].unique() if (x not in process_list)&(pd.notna(x))])
process_map = {x:count+0 for count,x in enumerate(process_list)}

process_map

{'ML Project': 0,
 'Goal Setting': 1,
 'Problem Definition': 2,
 'Data Collection': 3,
 'Data Preparation': 4,
 'Exploratory Data Analysis': 5,
 'Feature Engineering': 6,
 'Feature Selection': 7,
 'Model Selection': 8,
 'Training': 9,
 'Hyperparameter Tuning': 10,
 'Model Evaluation': 11,
 'Validation/ Model Selection': 12,
 'Model Interpretability': 13,
 'Deployment': 14,
 'Monitoring': 15,
 'Bias, Fairness, and Ethics': 16,
 'Process Development': 17,
 'Clustering': 18,
 'RAG': 19,
 'General Notes': 20,
 'Organization': 21,
 nan: 22,
 'ML Model': 23,
 'Mathematics': 24,
 'Algorithm Classification': 25,
 'Tooling': 26,
 'Validation': 27,
 'ML Definitions': 28,
 'Behavioural Economics': 29,
 'Theorem': 30,
 'Data Engineering': 31,
 'Graph Theory': 32,
 'Machine Learning Operations': 33,
 'Model Type': 34,
 'Generative AI': 35,
 'Langchain': 36,
 'Python': 37,
 'General Definition': 38}

In [24]:

# Step 2 Create a Unique Classification List
categorization_list = [
    'Definition','Guiding Principle','Consideration','Process Step','Procedure','Expected Outcomes','Parameter','Algorithm']

categorization_list.extend([x for x in notes_df['Categorization'].unique() if (x not in categorization_list)&(pd.notna(x))])
categorization_list.extend([x for x in definition_df['Categorization'].unique() if (x not in categorization_list)&(pd.notna(x))])
categorization_map = {x:count+0 for count,x in enumerate(categorization_list)}

final_df = pd.concat([notes_df,definition_df])
final_df['PRO_ORDER'] = final_df['Process'].map(process_map)
final_df['CAT_ORDER'] = final_df['Categorization'].map(categorization_map)

final_df = final_df.sort_values(['PRO_ORDER','CAT_ORDER'])
final_df.drop(['PRO_ORDER','CAT_ORDER'],axis=1,inplace=True)

['ML Project',
 'Goal Setting',
 'Problem Definition',
 'Data Collection',
 'Data Preparation',
 'Exploratory Data Analysis',
 'Feature Engineering',
 'Feature Selection',
 'Model Selection',
 'Training',
 'Hyperparameter Tuning',
 'Model Evaluation',
 'Validation/ Model Selection',
 'Model Interpretability',
 'Deployment',
 'Monitoring',
 'Bias, Fairness, and Ethics',
 'Process Development',
 'Clustering',
 'RAG',
 'General Notes',
 'Organization',
 nan]

In [29]:
from dict_definitions import links



d

,Process,Categorization,Word,Definition
0,ML Project,Process Step,Goal Setting,NaN
1,ML Project,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th..."
2,ML Project,Process Step,Data Collection,"Data collection involves identifying, sourcing..."
3,ML Project,Process Step,Data Preparation,Data preparation focuses on cleaning and struc...
4,ML Project,Process Step,Exploratory Data Analysis,Foundational step in ensuring the success of a...
...,...,...,...,...
423,Generative AI,Tooling,LangChain,Framework designed to simplify the development...
424,Langchain,Definition,Prompt templates,Key concept in LangChain. They help translate ...
425,Python,Guiding Principle,Python Function Naming Convention,Pythonic function naming follows the snake_cas...
426,Python,Guiding Principle,Data Set Column Naming,Dataset column naming should use ALL_CAPS_SNAK...


In [30]:
definition_df[definition_df['Process']=='ML Model']

,Process,Categorization,Word,Definition
13,ML Model,Guiding Principle,Machine Learning,Using a process to enable computers to iterati...
14,ML Model,Guiding Principle,Optimization,Optimization is the process of finding the bes...
15,ML Model,Pipeline Stage,Model Summarization / Model Interpretability,Model summarization and interpretability aim t...
16,ML Model,Pipeline Stage,Monitoring,Monitoring tracks model performance and behavi...
26,ML Model,Algorithm,Ada,Boosting technique that combines multiple weak...
...,...,...,...,...
371,ML Model,Algorithm,Hard Voting,Hard voting is an ensemble decision strategy w...
372,ML Model,Algorithm,Soft Voting,Soft voting is an ensemble technique where cla...
403,ML Model,Model Type,Assessment,Assessment refers to techniques that do not le...
432,ML Model,Algorithm,Naive Bayes,Naive Bayes is a family of probabilistic machi...


In [19]:
# IN ORDER TO CREATE THE FILE





from dict_processing import dict_to_dot_py

dict_defitions_dot_py()

    

In [ ]:
def create_parameter_list(categorization):
    
    '''
    Function Which Utilizes a Google Document Sheet to extract all records from the Column Categorization to generate a list of values, with
    populated Definitions. As reminder, this process utilizes and is reliant on the update of the Google Sheet, which if not completed can
    be conducted by function ________. 

    Parameters:
        categorization(str): Word which will be used to filter Dataframe to extract a specifically defined list.

        
    
    '''

    # Import Modified Notes. 
    

In [ ]:
function_fields = ['date_created','date_last_modified','classification','sub_classification','usage']

notes_default = ['Goal','Approach','Important to Remember','Lessons Learnt',"Algorithm",'Function','Constraint','Procedure']

In [24]:
from shared_folder import text_file_import,parse_dot_py_file

location = '/Users/derekdewald/Documents/Python/Github_Repo/d_py_functions/'
d = text_file_import(f"{location}feature_engineering.py")
a,b = parse_dot_py_file(d)

In [25]:
a

,Function,Purpose,Parameters,Returns,date_created,date_last_modified,classification,sub_classification,usage
0,binary_complex_equivlance,Function Used Test the Equivalency of Two Colu...,"[df, col, col1, new_column_name, eq, ne]",Object Type,10-Jun-26,10-Jun-26,TBD,TBD\ntest: Does this work?,Example Function Call
1,binary_column_creator,Function to Create a Binary Flag.\nUpdated to ...,"[df, column_name, new_column_name, value, calc...",None,None,None,None,None,None


In [26]:
b

,Function,Parameters,Type,Definition
0,binary_complex_equivlance,df,dataframe,Dataframe
1,binary_complex_equivlance,col,str,Name of Column to Test
2,binary_complex_equivlance,col1,str,Name of Other Column to Test
3,binary_complex_equivlance,new_column_name,str,Name of New Column to be used as Flag. If not ...
4,binary_complex_equivlance,eq,int,Value to Include to denote Match. Default is 1...
5,binary_complex_equivlance,ne,int,Value to Include to denote Not Match. Default ...
6,binary_column_creator,df,,
7,binary_column_creator,column_name,,
8,binary_column_creator,new_column_name,,
9,binary_column_creator,value,,
